# Bank marketing: prediction, cost, and capacity

Select clients under a forecast call-time budget, then compare the plan with actual historical call duration.

This walkthrough reads the exported results from the reproduced Python experiment. It does not train models in its cells. To regenerate the results, run the command below from the repository root, then rerun these cells.

```bash
python -m decision_analytics.bank
```

See [case README](README.md) for the full protocol and interpretation.

In [1]:
from pathlib import Path
import pandas as pd

ROOT = next(p for p in [Path.cwd(), *Path.cwd().parents]
            if (p / 'pyproject.toml').exists() and (p / 'src/decision_analytics').exists())
CASE = ROOT / 'projects' / 'bank-marketing'

## Subscription models

The classifier is selected by validation Brier score. The final row evaluates the selected model on the historical test rows.

In [2]:
pd.read_csv(CASE / 'results/classification.csv').round(5)

,model,split,roc_auc,average_precision,brier_score
0,Logistic regression,validation,0.79512,0.44630,0.07886
1,Random forest,validation,0.79940,0.46600,0.11510
2,Calibrated random forest,validation,0.80125,0.46589,0.07747
3,Calibrated random forest,test,0.80916,0.48864,0.07470


## Duration baselines

RMSE favors the training mean. MAE favors the log forest, illustrating why the error metric should match the decision objective.

In [3]:
pd.read_csv(CASE / 'results/duration.csv').round(4)

,model,split,mae_seconds,rmse_seconds,r2
0,Training mean,validation,170.6486,254.4796,-0.0000
1,Log-linear regression,validation,154.1592,264.5815,-0.0810
2,Log-random forest,validation,153.3469,262.5128,-0.0641
3,Training mean,test,173.1853,260.4891,-0.0000


## Forecast budgets and actual budgets

Keep the original duration forest as an experimental policy. Buffer values were selected using validation rows, before applying them to test rows.

In [4]:
policies = pd.read_csv(CASE / 'results/policies_test.csv')
focus = policies[(policies.scenario == 'baseline') & (policies.duration_model == 'Log-random forest') & policies.policy.isin(['Probability ranking', 'Duration-aware ranking'])]
focus[['policy', 'variant', 'buffer', 'contacts', 'predicted_minutes', 'actual_minutes', 'actual_overrun_pct']].round(3)

,policy,variant,buffer,contacts,predicted_minutes,actual_minutes,actual_overrun_pct
17,Probability ranking,unbuffered,1.0,776,2998.461,3893.450,29.782
18,Duration-aware ranking,unbuffered,1.0,875,2994.565,4051.450,35.048
19,Probability ranking,validation-selected buffer,0.8,602,2395.836,3110.117,3.671
20,Duration-aware ranking,validation-selected buffer,0.7,624,2097.432,2828.400,0.000


## Decision visualization

![Case results](figures/calibration_and_capacity.svg)

## Limits

These are retrospective random-split results on historical contacted customers. Profit uses assumptions, not causal lift. Predicted feasibility does not guarantee actual feasibility, and the original test rows had already been inspected in coursework.